In [1]:
import snowflake.connector
import pandas as pd

In [ ]:
# %pip install snowflake-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [4]:
conn = snowflake.connector.connect(
    user='ABOOD36',
    password='HezMEJkBh2xZ9tx',
    account='itcoyes-gf39385',   # from your URL
    warehouse='TOURISM_WH',
    database='TOURISM_DB',
    schema='GOLD'
)

query = """
SELECT
    f.location_id,
    f.date,
    f.timestamp,
    f.occupancy_rate,
    f.cpi,
    f.anomaly_flag,
    l.location_name,
    l.city,
    l.location_type,
    l.max_capacity,
    d.is_weekend,
    d.calendar_season,
    d.school_period,
    d.month,
    d.quarter,
    q.popularity_tier
FROM GOLD.FCT_CROWD_UNIFIED f
JOIN GOLD.DIM_LOCATION l ON f.location_id = l.location_id
JOIN GOLD.DIM_DATE d ON f.date = d.date
JOIN GOLD.LOCATION_QUALITY_SCORE q ON f.location_id = q.location_id
"""

df = pd.read_sql(query, conn)
print(df.shape)
df.head()

D:\Office\Temp\ipykernel_51792\4253994983.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


(586360, 16)


,LOCATION_ID,DATE,TIMESTAMP,OCCUPANCY_RATE,CPI,ANOMALY_FLAG,LOCATION_NAME,CITY,LOCATION_TYPE,MAX_CAPACITY,IS_WEEKEND,CALENDAR_SEASON,SCHOOL_PERIOD,MONTH,QUARTER,POPULARITY_TIER
0,SHM_01,2023-12-11,2023-12-11 11:00:00,1.000,0.900,True,Ras Mohammed,Sharm El Sheikh,Nature,5000,False,Winter,School Days,12,4,Tier1
1,SHM_01,2024-01-06,2024-01-06 17:00:00,0.989,0.942,True,Ras Mohammed,Sharm El Sheikh,Nature,5000,True,Winter,Midyear Vacation,1,1,Tier1
2,SHM_01,2023-09-15,2023-09-15 08:00:00,0.672,0.697,False,Ras Mohammed,Sharm El Sheikh,Nature,5000,False,Autumn,School Days,9,3,Tier1
3,SHM_01,2024-01-11,2024-01-11 16:00:00,1.000,0.922,True,Ras Mohammed,Sharm El Sheikh,Nature,5000,False,Winter,Midyear Vacation,1,1,Tier1
4,SHM_01,2024-05-09,2024-05-09 13:00:00,1.000,0.875,True,Ras Mohammed,Sharm El Sheikh,Nature,5000,False,Spring,School Days,5,2,Tier1


In [ ]:
# df = df.drop(columns=['ISLAMIC_HOLIDAY'])

In [5]:
print(df.shape)
print(df.isnull().sum())
print(df['OCCUPANCY_RATE'].describe())

(586360, 16)
LOCATION_ID        0
DATE               0
TIMESTAMP          0
OCCUPANCY_RATE     0
CPI                0
ANOMALY_FLAG       0
LOCATION_NAME      0
CITY               0
LOCATION_TYPE      0
MAX_CAPACITY       0
IS_WEEKEND         0
CALENDAR_SEASON    0
SCHOOL_PERIOD      0
MONTH              0
QUARTER            0
POPULARITY_TIER    0
dtype: int64
count    586360.000000
mean          0.536450
std           0.291274
min           0.031000
25%           0.289000
50%           0.483000
75%           0.784000
max           1.000000
Name: OCCUPANCY_RATE, dtype: float64


In [6]:
df.columns.tolist()

['LOCATION_ID',
 'DATE',
 'TIMESTAMP',
 'OCCUPANCY_RATE',
 'CPI',
 'ANOMALY_FLAG',
 'LOCATION_NAME',
 'CITY',
 'LOCATION_TYPE',
 'MAX_CAPACITY',
 'IS_WEEKEND',
 'CALENDAR_SEASON',
 'SCHOOL_PERIOD',
 'MONTH',
 'QUARTER',
 'POPULARITY_TIER']

In [7]:
# ---- 3. Make sure TIMESTAMP is a real datetime ----
df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'])

# ---- 4. Feature engineering: time-based features ----
#adding a columns for the hour
df['hour'] = df['TIMESTAMP'].dt.hour
#changes day of the week to numbers
df['day_of_week_num'] = df['TIMESTAMP'].dt.dayofweek  # 0=Monday


# cyclical encoding so hour 23 and hour 0 are "close" to the model
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# ---- 5. Encode categorical columns as numbers ----
categorical_cols = [
    'LOCATION_ID',
    'CITY',
    'LOCATION_TYPE',
    'POPULARITY_TIER',
    'SCHOOL_PERIOD',
    'CALENDAR_SEASON'
]
encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col + '_encoded'] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
# ---- 6. Pick final feature columns ----
feature_cols = [
    'hour_sin',
    'hour_cos',

    'MONTH',
    'QUARTER',

    'IS_WEEKEND',

    'SCHOOL_PERIOD_encoded',
    'CALENDAR_SEASON_encoded',

    'LOCATION_ID_encoded',
    'CITY_encoded',
    'LOCATION_TYPE_encoded',
    'POPULARITY_TIER_encoded',

    'MAX_CAPACITY'
]
X = df[feature_cols]
y = df['OCCUPANCY_RATE']

# ---- 7. Train/test split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(469088, 12) (117272, 12)


In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---- Train the model ----
model = RandomForestRegressor(
    n_estimators=100,   # number of trees
    random_state=42,    # reproducibility
    n_jobs=-1           # use all CPU cores, trains faster
)

model.fit(X_train, y_train)


#

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [9]:
import joblib

joblib.dump(model, "saved_model.pkl")
joblib.dump(encoders, "encoders.pkl")

print("Model saved successfully!")



Model saved successfully!


In [10]:
 #---- Evaluate on the test set ----
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

MAE  : 0.0468
RMSE : 0.0754
R²   : 0.9331


In [11]:
print(df[['CPI', 'OCCUPANCY_RATE']].corr())

                    CPI  OCCUPANCY_RATE
CPI             1.00000         0.97819
OCCUPANCY_RATE  0.97819         1.00000


In [12]:
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importances)

                    feature  importance
10  POPULARITY_TIER_encoded    0.477388
0                  hour_sin    0.118523
7       LOCATION_ID_encoded    0.089465
1                  hour_cos    0.071069
8              CITY_encoded    0.069360
9     LOCATION_TYPE_encoded    0.038473
2                     MONTH    0.037404
5     SCHOOL_PERIOD_encoded    0.037265
11             MAX_CAPACITY    0.027599
6   CALENDAR_SEASON_encoded    0.016499
3                   QUARTER    0.012427
4                IS_WEEKEND    0.004527


In [13]:
print(df.groupby('POPULARITY_TIER')['OCCUPANCY_RATE'].mean())

POPULARITY_TIER
Tier1    0.693897
Tier2    0.318882
Tier3    0.186184
Name: OCCUPANCY_RATE, dtype: float64


In [14]:
from datetime import timedelta

# ---- 1. Get "tomorrow" relative to your data, not real time ----
max_date_query = "SELECT MAX(date) as max_date FROM GOLD.DIM_DATE"
max_date_result = pd.read_sql(max_date_query, conn)
max_date = pd.to_datetime(max_date_result['MAX_DATE'].iloc[0])
tomorrow = (max_date + timedelta(days=1)).date()

# ---- 2. Get unique locations with their static info ----
locations = df[['LOCATION_ID', 'LOCATION_ID_encoded',
                 'CITY', 'CITY_encoded',
                 'LOCATION_TYPE', 'LOCATION_TYPE_encoded',
                 'POPULARITY_TIER', 'POPULARITY_TIER_encoded',
                 'MAX_CAPACITY']].drop_duplicates(subset=['LOCATION_ID'])

# ---- 3. Calculate tomorrow's calendar info ourselves (DIM_DATE doesn't cover this date) ----
is_weekend_tomorrow = 1 if tomorrow.weekday() >= 5 else 0

# Build a month -> school_period mapping from your real data (do this once)
month_to_period = df[['MONTH', 'SCHOOL_PERIOD']].drop_duplicates().set_index('MONTH')['SCHOOL_PERIOD'].to_dict()

# Then, whenever "tomorrow" changes, look it up automatically:
tomorrow_month = tomorrow.month
school_period_tomorrow = month_to_period[tomorrow_month]
school_period_encoded_tomorrow = encoders['SCHOOL_PERIOD'].transform([school_period_tomorrow])[0]

# ---- 4. Build one row per (location, hour) for tomorrow ----
rows = []
month_tomorrow = tomorrow.month

quarter_tomorrow = (month_tomorrow - 1) // 3 + 1

if month_tomorrow in [12, 1, 2]:
    season = "Winter"
elif month_tomorrow in [3, 4, 5]:
    season = "Spring"
elif month_tomorrow in [6, 7, 8]:
    season = "Summer"
else:
    season = "Autumn"

season_encoded = encoders["CALENDAR_SEASON"].transform([season])[0]

for _, loc in locations.iterrows():
    for hour in range(24):
        hour_sin = np.sin(2 * np.pi * hour / 24)
        hour_cos = np.cos(2 * np.pi * hour / 24)
        rows.append({
            'hour_sin': hour_sin,
            'hour_cos': hour_cos,
            'IS_WEEKEND': is_weekend_tomorrow,
            'SCHOOL_PERIOD_encoded': school_period_encoded_tomorrow,
            'LOCATION_ID_encoded': loc['LOCATION_ID_encoded'],
            'CITY_encoded': loc['CITY_encoded'],
            'LOCATION_TYPE_encoded': loc['LOCATION_TYPE_encoded'],
            'POPULARITY_TIER_encoded': loc['POPULARITY_TIER_encoded'],
            'MAX_CAPACITY': loc['MAX_CAPACITY'],
            'location_id': loc['LOCATION_ID'],
            'city': loc['CITY'],
            'hour': hour,
            'MONTH': month_tomorrow,
            'QUARTER': quarter_tomorrow,
            'CALENDAR_SEASON_encoded': season_encoded,
        })

tomorrow_df = pd.DataFrame(rows)
print(tomorrow_df.shape)
tomorrow_df.head()

D:\Office\Temp\ipykernel_51792\1069702432.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  max_date_result = pd.read_sql(max_date_query, conn)


(1080, 15)


,hour_sin,hour_cos,IS_WEEKEND,SCHOOL_PERIOD_encoded,LOCATION_ID_encoded,CITY_encoded,LOCATION_TYPE_encoded,POPULARITY_TIER_encoded,MAX_CAPACITY,location_id,city,hour,MONTH,QUARTER,CALENDAR_SEASON_encoded
0,0.000000,1.000000,0,0,40,8,5,0,5000,SHM_01,Sharm El Sheikh,0,1,1,3
1,0.258819,0.965926,0,0,40,8,5,0,5000,SHM_01,Sharm El Sheikh,1,1,1,3
2,0.500000,0.866025,0,0,40,8,5,0,5000,SHM_01,Sharm El Sheikh,2,1,1,3
3,0.707107,0.707107,0,0,40,8,5,0,5000,SHM_01,Sharm El Sheikh,3,1,1,3
4,0.866025,0.500000,0,0,40,8,5,0,5000,SHM_01,Sharm El Sheikh,4,1,1,3


In [15]:
# ---- Predict occupancy for tomorrow ----
X_tomorrow = tomorrow_df[feature_cols]  # same feature columns used in training

tomorrow_df['predicted_occupancy'] = model.predict(X_tomorrow)

# quick look
tomorrow_df[['location_id', 'city', 'hour', 'predicted_occupancy']].sort_values(
    ['location_id', 'hour']
).head(30)

,location_id,city,hour,predicted_occupancy
1032,ALX_01,Alexandria,0,0.376237
1033,ALX_01,Alexandria,1,0.381772
1034,ALX_01,Alexandria,2,0.372310
1035,ALX_01,Alexandria,3,0.382257
1036,ALX_01,Alexandria,4,0.373936
1037,ALX_01,Alexandria,5,0.379424
1038,ALX_01,Alexandria,6,0.272330
1039,ALX_01,Alexandria,7,0.272330
1040,ALX_01,Alexandria,8,0.272330
1041,ALX_01,Alexandria,9,0.272166


In [16]:
# Overcrowded locations 1PM-3PM
overcrowded = tomorrow_df[
    (tomorrow_df['hour'].between(13, 15)) &
    (tomorrow_df['predicted_occupancy'] > 0.85)
]
print(overcrowded[['location_id', 'city', 'hour', 'predicted_occupancy']])

# Best time in Cairo
cairo = tomorrow_df[tomorrow_df['city'] == 'Cairo']
best_hour = cairo.groupby('hour')['predicted_occupancy'].mean().idxmin()
print("Best hour in Cairo:", best_hour)

     location_id             city  hour  predicted_occupancy
13        SHM_01  Sharm El Sheikh    13             1.000000
14        SHM_01  Sharm El Sheikh    14             1.000000
15        SHM_01  Sharm El Sheikh    15             0.998965
37        FYM_04           Fayoum    13             1.000000
38        FYM_04           Fayoum    14             1.000000
...          ...              ...   ...                  ...
1022      GIZ_03             Giza    14             1.000000
1023      GIZ_03             Giza    15             1.000000
1069      LUX_01            Luxor    13             1.000000
1070      LUX_01            Luxor    14             0.997706
1071      LUX_01            Luxor    15             0.998960

[64 rows x 4 columns]
Best hour in Cairo: 2


In [ ]:
#tomorrow_df.to_csv('tomorrow_predictions.csv', index=False)

In [ ]:
# %pip install "snowflake-connector-python[pandas]"


   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
    --------------------------------------- 0.3/11.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.0 MB 3.7 MB/s eta 0:00:03
   ----- ---------------------------------- 1.6/11.0 MB 3.3 MB/s eta 0:00:03
   -------- ------------------------------- 2.4/11.0 MB 3.3 MB/s eta 0:00:03
   ----------- ---------------------------- 3.1/11.0 MB 3.4 MB/s eta 0:00:03
   -------------- ------------------------- 3.9/11.0 MB 3.5 MB/s eta 0:00:03
   ----------------- ---------------------- 4.7/11.0 MB 3.6 MB/s eta 0:00:02
   -------------------- ------------------- 5.5/11.0 MB 3.5 MB/s eta 0:00:02
   ---------------------- ----------------- 6.3/11.0 MB 3.6 MB/s eta 0:00:02
   ------------------------- -------------- 7.1/11.0 MB 3.6 MB/s eta 0:00:02
   ---------------------------- ----------- 7.9/11.0 MB 3.6 MB/s eta 0:00:01
   ------------------------------ --------- 8.4/11.0 MB 3.6 MB/s eta 0:00:01
   ----------

  You can safely remove it manually.

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# %pip install pyarrow


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from snowflake.connector.pandas_tools import write_pandas

success, nchunks, nrows, _ = write_pandas(
    conn,
    tomorrow_df,
    "TOMORROW_PREDICTIONS",
    schema="GOLD",
    auto_create_table=True
)

In [18]:
print(success, nrows, "rows written")


True 1080 rows written
